In [12]:
import math

# --- CORE MINIMAX ALGORITHM ---

def terminal_test(state):
    """Returns True if the game is over (win/loss/draw), else False."""
    board, _ = state
    if check_winner(board) is not None:
        return True
    for row in board:
        if '_' in row:
            return False
    return True


def utility(state, max_player, min_player):
    """
    UTILITY(state)
    +1 if MAX wins, -1 if MIN wins, 0 for draw.
    """
    board, _ = state
    winner = check_winner(board)
    if winner == max_player:
        return 1
    elif winner == min_player:
        return -1
    else:
        return 0


def actions(state):
    """Returns list of legal actions (r, c) available in current state."""
    board, _ = state
    moves = []
    for r in range(3):
        for c in range(3):
            if board[r][c] == '_':
                moves.append((r, c))
    return moves


def result(state, action, max_player, min_player):
    """Returns new state resulting from playing action in current state."""
    board, current_player = state
    r, c = action
    new_board = [row[:] for row in board]
    new_board[r][c] = current_player
    next_player = min_player if current_player == max_player else max_player
    return (new_board, next_player)


def max_value(state, max_player, min_player):
    """MAX-VALUE(state)"""
    if terminal_test(state):
        return utility(state, max_player, min_player)

    v = -math.inf
    for action in actions(state):
        next_state = result(state, action, max_player, min_player)
        v = max(v, min_value(next_state, max_player, min_player))
    return v


def min_value(state, max_player, min_player):
    """MIN-VALUE(state)"""
    if terminal_test(state):
        return utility(state, max_player, min_player)

    v = math.inf
    for action in actions(state):
        next_state = result(state, action, max_player, min_player)
        v = min(v, max_value(next_state, max_player, min_player))
    return v


def minimax_decision(state, max_player, min_player):
    """
    MINIMAX-DECISION(state)
    Returns best move for computer player using Minimax recursion.
    """
    best_action = None
    best_val = -math.inf

    for action in actions(state):
        next_state = result(state, action, max_player, min_player)
        val = min_value(next_state, max_player, min_player)
        if val > best_val:
            best_val = val
            best_action = action

    return best_action


# --- HELPER & GAME UI FUNCTIONS ---

def check_winner(board):
    """Checks board for 3-in-a-row winner ('X' or 'O')."""
    for i in range(3):
        if board[i][0] != '_' and board[i][0] == board[i][1] == board[i][2]:
            return board[i][0]
        if board[0][i] != '_' and board[0][i] == board[1][i] == board[2][i]:
            return board[0][i]
    if board[0][0] != '_' and board[0][0] == board[1][1] == board[2][2]:
        return board[0][0]
    if board[0][2] != '_' and board[0][2] == board[1][1] == board[2][0]:
        return board[0][2]
    return None


def print_board(board):
    """Displays current board state."""
    print("\n  1 2 3")
    for idx, row in enumerate(board, start=1):
        print(f"{idx} " + " ".join(row))
    print()


def get_human_move(board):
    """Gets and validates row, column input from the player."""
    while True:
        try:
            inp = input("Enter your move as 'row col' (e.g., 1 2): ").strip().split()
            if len(inp) != 2:
                print("Invalid input format. Provide two numbers separated by a space.")
                continue
            r, c = int(inp[0]) - 1, int(inp[1]) - 1
            if 0 <= r <= 2 and 0 <= c <= 2:
                if board[r][c] == '_':
                    return (r, c)
                else:
                    print("Position already taken! Pick an empty slot.")
            else:
                print("Position out of range! Use values 1, 2, or 3.")
        except ValueError:
            print("Invalid input! Please enter integers.")


# --- MAIN INTERACTIVE GAME LOOP ---

def play_game():
    board = [['_' for _ in range(3)] for _ in range(3)]

    print("=== TIC-TAC-TOE vs AI (MINIMAX) ===")

    # Choose Symbol
    human_symbol = ""
    while human_symbol not in ['X', 'O']:
        human_symbol = input("Choose your symbol (X/O): ").strip().upper()
    computer_symbol = 'O' if human_symbol == 'X' else 'X'

    # Choose who starts
    first = ""
    while first not in ['H', 'C']:
        first = input("Who goes first? Human (H) or Computer (C): ").strip().upper()
    current_player = human_symbol if first == 'H' else computer_symbol

    print_board(board)

    # Main Play Loop
    while True:
        state = (board, current_player)

        if terminal_test(state):
            winner = check_winner(board)
            if winner == human_symbol:
                print("Congratulations! You won!")
            elif winner == computer_symbol:
                print("Computer wins! Better luck next time.")
            else:
                print("It's a draw!")
            break

        if current_player == human_symbol:
            print("Your turn:")
            r, c = get_human_move(board)
            board[r][c] = human_symbol
            print_board(board)
            current_player = computer_symbol
        else:
            print("Computer is calculating move...")
            move = minimax_decision(state, computer_symbol, human_symbol)
            board[move[0]][move[1]] = computer_symbol
            print(f"Computer places {computer_symbol} at position ({move[0]+1}, {move[1]+1})")
            print_board(board)
            current_player = human_symbol


if __name__ == "__main__":
    play_game()

=== TIC-TAC-TOE vs AI (MINIMAX) ===
Choose your symbol (X/O): x
Who goes first? Human (H) or Computer (C): h

  1 2 3
1 _ _ _
2 _ _ _
3 _ _ _

Your turn:
Enter your move as 'row col' (e.g., 1 2): 1 1

  1 2 3
1 X _ _
2 _ _ _
3 _ _ _

Computer is calculating move...
Computer places O at position (2, 2)

  1 2 3
1 X _ _
2 _ O _
3 _ _ _

Your turn:
Enter your move as 'row col' (e.g., 1 2): 2 1

  1 2 3
1 X _ _
2 X O _
3 _ _ _

Computer is calculating move...
Computer places O at position (3, 1)

  1 2 3
1 X _ _
2 X O _
3 O _ _

Your turn:
Enter your move as 'row col' (e.g., 1 2): 1 3

  1 2 3
1 X _ X
2 X O _
3 O _ _

Computer is calculating move...
Computer places O at position (1, 2)

  1 2 3
1 X O X
2 X O _
3 O _ _

Your turn:
Enter your move as 'row col' (e.g., 1 2): 3 2

  1 2 3
1 X O X
2 X O _
3 O X _

Computer is calculating move...
Computer places O at position (2, 3)

  1 2 3
1 X O X
2 X O O
3 O X _

Your turn:
Enter your move as 'row col' (e.g., 1 2): 3 3

  1 2 3
1 X O X
2 X O O

In [13]:
import heapq

# Goal state standard representation (0 represents the blank space)
GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def __lt__(self, other):
        # Tie-breaker for the priority queue based on path cost
        return self.path_cost < other.path_cost


def position_of_blank(state):
    """Finds the (row, col) position of the blank tile (0)."""
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c
    return None


def position(tile, state):
    """Finds the (row, col) position of a given tile in state."""
    for r in range(3):
        for c in range(3):
            if state[r][c] == tile:
                return r, c
    return None


def manhattan_distance(state):
    """MANHATTAN-DISTANCE(state)"""
    distance = 0
    for r in range(3):
        for c in range(3):
            t = state[r][c]
            if t != 0:
                goal_r, goal_c = position(t, GOAL_STATE)
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance


def actions(state):
    """Returns valid actions (UP, DOWN, LEFT, RIGHT) based on blank position."""
    r, c = position_of_blank(state)
    valid_actions = []
    if r > 0:
        valid_actions.append('UP')
    if r < 2:
        valid_actions.append('DOWN')
    if c > 0:
        valid_actions.append('LEFT')
    if c < 2:
        valid_actions.append('RIGHT')
    return valid_actions


def child_node(problem, node, action):
    """CHILD-NODE(problem, node, action)"""
    # Create mutable copy of current board state
    state = [list(row) for row in node.state]
    r, c = position_of_blank(node.state)

    # Move blank tile based on action
    if action == 'UP':
        state[r][c], state[r - 1][c] = state[r - 1][c], state[r][c]
    elif action == 'DOWN':
        state[r][c], state[r + 1][c] = state[r + 1][c], state[r][c]
    elif action == 'LEFT':
        state[r][c], state[r][c - 1] = state[r][c - 1], state[r][c]
    elif action == 'RIGHT':
        state[r][c], state[r][c + 1] = state[r][c + 1], state[r][c]

    # Convert state back to tuple for hashability in sets/queues
    new_state = tuple(tuple(row) for row in state)

    return Node(
        state=new_state,
        parent=node,
        action=action,
        path_cost=node.path_cost + 1
    )


def goal_test(problem, state):
    """GOAL-TEST(problem, node.state)"""
    return state == GOAL_STATE


def solution(node):
    """Reconstructs the solution path from goal node back to root."""
    actions_list = []
    curr = node
    while curr.parent is not None:
        actions_list.append(curr.action)
        curr = curr.parent
    actions_list.reverse()
    return actions_list


def eight_puzzle_a_star(initial_state):
    """8-PUZZLE-A-STAR(problem)"""
    node = Node(initial_state)

    # Priority Queue entries stored as: (f_score, counter, node)
    # Counter breaks ties when f_scores are equal
    frontier = []
    counter = 0

    g_node = 0
    f_node = g_node + manhattan_distance(node.state)

    heapq.heappush(frontier, (f_node, counter, node))

    explored = set()

    while len(frontier) > 0:
        f_val, _, node = heapq.heappop(frontier)

        if goal_test(None, node.state):
            return solution(node)

        if node.state not in explored:
            explored.add(node.state)

            for action in actions(node.state):
                child = child_node(None, node, action)
                g_child = node.path_cost + 1
                f_child = g_child + manhattan_distance(child.state)

                if child.state not in explored:
                    counter += 1
                    heapq.heappush(frontier, (f_child, counter, child))

    return None


def print_board(state):
    """Prints board formatted nicely with blank as '_'."""
    for row in state:
        print(" ".join(str(val) if val != 0 else '_' for val in row))
    print()


# --- EXECUTION & DEMO ---

if __name__ == "__main__":
    # Example Initial State (Solvable 8-Puzzle setup)
    initial_board = (
        (1, 2, 3),
        (0, 4, 6),
        (7, 5, 8)
    )

    print("Initial Board State:")
    print_board(initial_board)

    print("Solving using A* with Manhattan Distance...\n")
    path = eight_puzzle_a_star(initial_board)

    if path:
        print(f"Goal reached in {len(path)} steps!")
        print("Action Sequence:", " -> ".join(path))
    else:
        print("No solution found.")

Initial Board State:
1 2 3
_ 4 6
7 5 8

Solving using A* with Manhattan Distance...

Goal reached in 3 steps!
Action Sequence: RIGHT -> DOWN -> RIGHT


In [14]:
import heapq

# Goal state standard representation (0 represents the blank space)
GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost


def position_of_blank(state):
    """POSITION-OF-BLANK(state)"""
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c
    return None


def position(tile, state):
    """POSITION(t, state)"""
    for r in range(3):
        for c in range(3):
            if state[r][c] == tile:
                return r, c
    return None


def manhattan_distance(state):
    """MANHATTAN-DISTANCE(state)"""
    distance = 0
    for r in range(3):
        for c in range(3):
            t = state[r][c]
            if t != 0:
                goal_r, goal_c = position(t, GOAL_STATE)
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance


def actions(state):
    """ACTIONS(node.state)"""
    r, c = position_of_blank(state)
    valid_actions = []
    if r > 0:
        valid_actions.append('UP')
    if r < 2:
        valid_actions.append('DOWN')
    if c > 0:
        valid_actions.append('LEFT')
    if c < 2:
        valid_actions.append('RIGHT')
    return valid_actions


def child_node(problem, node, action):
    """CHILD-NODE(problem, node, action)"""
    state = [list(row) for row in node.state]
    r, c = position_of_blank(node.state)

    if action == 'UP':
        state[r][c], state[r - 1][c] = state[r - 1][c], state[r][c]
    elif action == 'DOWN':
        state[r][c], state[r + 1][c] = state[r + 1][c], state[r][c]
    elif action == 'LEFT':
        state[r][c], state[r][c - 1] = state[r][c - 1], state[r][c]
    elif action == 'RIGHT':
        state[r][c], state[r][c + 1] = state[r][c + 1], state[r][c]

    new_state = tuple(tuple(row) for row in state)

    return Node(
        state=new_state,
        parent=node,
        action=action,
        path_cost=node.path_cost + 1
    )


def goal_test(problem, state):
    """GOAL-TEST(problem, node.state)"""
    return state == GOAL_STATE


def solution_nodes(node):
    """Reconstructs path nodes from start to goal."""
    path = []
    curr = node
    while curr is not None:
        path.append(curr)
        curr = curr.parent
    path.reverse()
    return path


def eight_puzzle_a_star(initial_state):
    """8-PUZZLE-A-STAR(problem)"""
    node = Node(initial_state)

    frontier = []
    counter = 0

    g_node = 0
    f_node = g_node + manhattan_distance(node.state)

    heapq.heappush(frontier, (f_node, counter, node))
    explored = set()

    while len(frontier) > 0:
        f_val, _, node = heapq.heappop(frontier)

        if goal_test(None, node.state):
            return solution_nodes(node)

        if node.state not in explored:
            explored.add(node.state)

            for action in actions(node.state):
                child = child_node(None, node, action)
                g_child = child.path_cost
                f_child = g_child + manhattan_distance(child.state)

                if child.state not in explored:
                    counter += 1
                    heapq.heappush(frontier, (f_child, counter, child))

    return None


def print_board(state):
    """Prints 3x3 state with '_' representing blank space."""
    for row in state:
        print(" ".join(str(val) if val != 0 else '_' for val in row))


def run_and_format_solution(initial_board, test_case_num):
    """Solves the state and prints matching test outputs."""
    path = eight_puzzle_a_star(initial_board)

    if not path:
        print("No solution found.")
        return

    cost = len(path) - 1

    if test_case_num in [1, 2]:
        for step_idx in range(1, len(path)):
            curr_node = path[step_idx]
            if test_case_num == 1:
                print(f"Move: {curr_node.action}")
            else:
                print(f"Move {step_idx}: {curr_node.action}")
            print_board(curr_node.state)
        print("Goal State Reached")
        print(f"Solution Cost = {cost}")

    elif test_case_num == 3:
        print("Solution Found")
        print("Goal State:")
        print_board(path[-1].state)
        print(f"Solution Cost = {cost}")


# --- EXECUTION ---

# Test Case 1: One-Move Solution
input_1 = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 0, 8)
)
run_and_format_solution(input_1, 1)

print()

# Test Case 2: Two-Move Solution
input_2 = (
    (1, 2, 3),
    (4, 0, 6),
    (7, 5, 8)
)
run_and_format_solution(input_2, 2)

print()

# Test Case 3: More Complex Solution
input_3 = (
    (1, 2, 3),
    (5, 0, 6),
    (4, 7, 8)
)
run_and_format_solution(input_3, 3)

Move: RIGHT
1 2 3
4 5 6
7 8 _
Goal State Reached
Solution Cost = 1

Move 1: DOWN
1 2 3
4 5 6
7 _ 8
Move 2: RIGHT
1 2 3
4 5 6
7 8 _
Goal State Reached
Solution Cost = 2

Solution Found
Goal State:
1 2 3
4 5 6
7 8 _
Solution Cost = 4


In [18]:
import heapq

GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost


def position_of_blank(state):
    """POSITION-OF-BLANK(state)"""
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c
    return None


def position(tile, state):
    """POSITION(t, state)"""
    for r in range(3):
        for c in range(3):
            if state[r][c] == tile:
                return r, c
    return None


def manhattan_distance(state):
    """MANHATTAN-DISTANCE(state)"""
    distance = 0
    for r in range(3):
        for c in range(3):
            t = state[r][c]
            if t != 0:
                goal_r, goal_c = position(t, GOAL_STATE)
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance


def actions(state):
    """ACTIONS(node.state)"""
    r, c = position_of_blank(state)
    valid_actions = []
    if r > 0:
        valid_actions.append('UP')
    if r < 2:
        valid_actions.append('DOWN')
    if c > 0:
        valid_actions.append('LEFT')
    if c < 2:
        valid_actions.append('RIGHT')
    return valid_actions


def child_node(problem, node, action):
    """CHILD-NODE(problem, node, action)"""
    state = [list(row) for row in node.state]
    r, c = position_of_blank(node.state)

    if action == 'UP':
        state[r][c], state[r - 1][c] = state[r - 1][c], state[r][c]
    elif action == 'DOWN':
        state[r][c], state[r + 1][c] = state[r + 1][c], state[r][c]
    elif action == 'LEFT':
        state[r][c], state[r][c - 1] = state[r][c - 1], state[r][c]
    elif action == 'RIGHT':
        state[r][c], state[r][c + 1] = state[r][c + 1], state[r][c]

    new_state = tuple(tuple(row) for row in state)

    return Node(
        state=new_state,
        parent=node,
        action=action,
        path_cost=node.path_cost + 1
    )


def goal_test(problem, state):
    """GOAL-TEST(problem, node.state)"""
    return state == GOAL_STATE


def solution_nodes(node):
    """Reconstructs path nodes from start to goal."""
    path = []
    curr = node
    while curr is not None:
        path.append(curr)
        curr = curr.parent
    path.reverse()
    return path


def eight_puzzle_a_star(initial_state):
    """8-PUZZLE-A-STAR(problem)"""
    node = Node(initial_state)

    frontier = []
    counter = 0

    g_node = 0
    f_node = g_node + manhattan_distance(node.state)

    heapq.heappush(frontier, (f_node, counter, node))
    explored = set()

    while len(frontier) > 0:
        f_val, _, node = heapq.heappop(frontier)

        if goal_test(None, node.state):
            return solution_nodes(node)

        if node.state not in explored:
            explored.add(node.state)

            for action in actions(node.state):
                child = child_node(None, node, action)
                g_child = child.path_cost
                f_child = g_child + manhattan_distance(child.state)

                if child.state not in explored:
                    counter += 1
                    heapq.heappush(frontier, (f_child, counter, child))

    return None


def print_board(state):
    """Prints 3x3 board state."""
    for row in state:
        print(" ".join(str(val) if val != 0 else '_' for val in row))


def is_solvable(state):
    """Checks inversion count to confirm if state is solvable."""
    flat_list = [tile for row in state for tile in row if tile != 0]
    inversions = 0
    for i in range(len(flat_list)):
        for j in range(i + 1, len(flat_list)):
            if flat_list[i] > flat_list[j]:
                inversions += 1
    # Solvable if inversions are even
    return inversions % 2 == 0


def get_user_board():
    """Reads and validates 3x3 board input from the user."""
    print("Enter the initial 8-puzzle board row by row.")
    print("Use numbers 1-8 for tiles and 0 or _ for the blank space.\n")

    board = []
    elements = set()

    for i in range(3):
        while True:
            row_str = input(f"Enter Row {i+1} (3 values separated by spaces): ").strip()
            tokens = row_str.split()
            if len(tokens) != 3:
                print("Error: Please enter exactly 3 values.")
                continue

            row_vals = []
            valid = True
            for t in tokens:
                if t == '_':
                    val = 0
                else:
                    try:
                        val = int(t)
                    except ValueError:
                        valid = False
                        break

                if val < 0 or val > 8 or val in elements:
                    valid = False
                    break
                row_vals.append(val)

            if not valid:
                print("Error: Invalid numbers or duplicate values detected. Use numbers 0-8.")
                continue

            for v in row_vals:
                elements.add(v)
            board.append(tuple(row_vals))
            break

    return tuple(board)


# --- MAIN INTERACTIVE PROGRAM ---

if __name__ == "__main__":
    initial_board = get_user_board()

    print("\nInitial State:")
    print_board(initial_board)
    print()

    if not is_solvable(initial_board):
        print("This puzzle state is un-solvable!")
    else:
        print("Solving puzzle using A* Search...\n")
        path = eight_puzzle_a_star(initial_board)

        if path:
            print("--- Solution Steps ---")
            for step_idx in range(1, len(path)):
                curr_node = path[step_idx]
                print(f"Move {step_idx}: {curr_node.action}")
                print_board(curr_node.state)
                print()

            print("Goal State Reached!")
            print(f"Solution Cost = {len(path) - 1}")
        else:
            print("No solution found.")

Enter the initial 8-puzzle board row by row.
Use numbers 1-8 for tiles and 0 or _ for the blank space.

Enter Row 1 (3 values separated by spaces): 1 2 3
Enter Row 2 (3 values separated by spaces): 5 _ 6
Enter Row 3 (3 values separated by spaces): 4 7 8

Initial State:
1 2 3
5 _ 6
4 7 8

Solving puzzle using A* Search...

--- Solution Steps ---
Move 1: LEFT
1 2 3
_ 5 6
4 7 8

Move 2: DOWN
1 2 3
4 5 6
_ 7 8

Move 3: RIGHT
1 2 3
4 5 6
7 _ 8

Move 4: RIGHT
1 2 3
4 5 6
7 8 _

Goal State Reached!
Solution Cost = 4
